In [18]:
#!/usr/bin/env python3
import pandas as pd
import requests
import time
import re
from pathlib import Path
from typing import Optional

# ---- config ----
EXCEL_PATH = Path("/content/drive/MyDrive/Lab Work/Bayesian_Graph_Metabolic_Paper/KEGG_Graph_Generation/KEGG_Prokaryote_Modules_with_Definitions.xlsx")
ID_COLUMN = "Module ID"
OUT_COLUMN = "Module_Expression"
FILL_MISSING_ONLY = True   # set to False to overwrite any existing values
REQUEST_TIMEOUT = 15
RETRY_ATTEMPTS = 4
RETRY_BACKOFF_SEC = 2.0
POLITE_DELAY_SEC = 0.3

In [19]:
# be nice to the KEGG server
KEGG_URL_TPL = "https://rest.kegg.jp/get/module:{mod_id}"

def normalize_mod_id(raw: str) -> Optional[str]:
    """Return 'M#####' or None if not a plausible KEGG module id."""
    if not isinstance(raw, str):
        return None
    raw = raw.strip()
    m = re.fullmatch(r"(?:(?:module:)?)(M\d{5})", raw, flags=re.IGNORECASE)
    return m.group(1).upper() if m else None

def fetch_kegg_entry(mod_id: str) -> Optional[str]:
    """Fetch the flat-file entry text for a module; return text or None on failure."""
    url = KEGG_URL_TPL.format(mod_id=mod_id)
    last_err = None
    for attempt in range(1, RETRY_ATTEMPTS + 1):
        try:
            r = requests.get(url, timeout=REQUEST_TIMEOUT)
            if r.status_code == 200 and r.text:
                return r.text
            # 404 or other status: keep last_err and retry a couple times
            last_err = RuntimeError(f"HTTP {r.status_code} from {url}")
        except requests.RequestException as e:
            last_err = e
        # exponential-ish backoff
        time.sleep(RETRY_BACKOFF_SEC * attempt)
    # final failure
    print(f"[WARN] Failed to fetch {mod_id}: {last_err}")
    return None

def parse_definition(flat_text: str) -> Optional[str]:
    """
    Parse KEGG flat file DEFINITION as multi-line text.
    Keep continuation lines and join them with '\n' (NOT spaces).
    """
    if not flat_text:
        return None

    lines = flat_text.splitlines()
    collecting = False
    payloads = []

    for line in lines:
        # KEGG: first 12 columns = field name area, rest is payload
        field = line[:12].strip()
        rest  = line[12:]

        if field == "DEFINITION":
            collecting = True
            payloads.append(rest.rstrip())        # keep spacing inside this line
        elif collecting:
            # Continuation lines have no new field and start with spaces
            if field == "":                       # still in DEFINITION
                payloads.append(rest.rstrip())
            else:
                break

    if not payloads:
        return None

    # CRITICAL: keep multi-line structure (top-level alternatives)
    # - strip only trailing whitespace per line (already done via rstrip)
    # - do NOT collapse inner spaces
    definition = "\n".join(payloads).strip("\n")
    return definition or None


def main():
    if not EXCEL_PATH.exists():
        raise FileNotFoundError(f"Excel file not found: {EXCEL_PATH}")

    df = pd.read_excel(EXCEL_PATH)

    if ID_COLUMN not in df.columns:
        raise KeyError(f"Required column '{ID_COLUMN}' not found in {EXCEL_PATH}")

    if OUT_COLUMN not in df.columns:
        df[OUT_COLUMN] = ""

    # Work row-by-row
    for idx, raw_id in df[ID_COLUMN].items():
        mod_id = normalize_mod_id(str(raw_id))
        if not mod_id:
            print(f"[INFO] Skipping row {idx}: invalid Module ID '{raw_id}'")
            continue

        # Skip if we only fill missing and there is already a value
        if FILL_MISSING_ONLY and isinstance(df.at[idx, OUT_COLUMN], str) and df.at[idx, OUT_COLUMN].strip():
            continue

        # Polite pacing
        time.sleep(POLITE_DELAY_SEC)

        entry_text = fetch_kegg_entry(mod_id)
        definition = parse_definition(entry_text) if entry_text else None

        if definition:
          df.at[idx, OUT_COLUMN] = definition
          # Debug proof
          if "\n" in definition:
              print(f"[OK] {mod_id}: multi-line ({definition.count('\\n')+1} lines)")
          else:
              print(f"[OK] {mod_id}: single line")


    # Save back to the same file (or change path if you want a new file)
    backup_path = EXCEL_PATH.with_suffix(".backup.xlsx")
    df.to_excel(EXCEL_PATH, index=False)
    print(f"Saved updated definitions to {EXCEL_PATH}")

if __name__ == "__main__":
    main()


[OK] M00001: single line
[OK] M00002: single line
[OK] M00003: single line
[OK] M00004: single line
[OK] M00005: single line
[OK] M00006: single line
[OK] M00007: single line
[OK] M00008: single line
[OK] M00009: single line
[OK] M00010: single line
[OK] M00011: single line
[OK] M00012: single line
[OK] M00015: single line
[OK] M00016: single line
[OK] M00017: single line
[OK] M00018: single line
[OK] M00019: single line
[OK] M00020: single line
[OK] M00021: single line
[OK] M00022: single line
[OK] M00023: single line
[OK] M00024: single line
[OK] M00025: single line
[OK] M00026: single line
[OK] M00027: single line
[OK] M00028: single line
[OK] M00031: multi-line (1 lines)
[OK] M00033: single line
[OK] M00034: single line
[OK] M00035: single line
[OK] M00036: single line
[OK] M00038: single line
[OK] M00040: single line
[OK] M00043: single line
[OK] M00044: single line
[OK] M00045: single line
[OK] M00046: single line
[OK] M00048: single line
[OK] M00049: single line
[OK] M00050: sin

In [ ]:
def _normalize(x):
    if pd.isna(x):
        return ""
    return str(x).strip()
df =pd.read_excel(EXCEL_PATH)
# apply normalization
df["Definition_norm"] = df["Definition"].apply(_normalize)
df["ModuleExpr_norm"] = df["Module_Expression"].apply(_normalize)

# find rows where they differ
diff_df = df[df["Definition_norm"] != df["ModuleExpr_norm"]]

# show only the relevant columns
result = diff_df[["Module ID", "Definition", "Module_Expression"]]

In [ ]:
result.shape

(23, 3)

In [ ]:
result[result['Module ID'] == "M00173"].values

array([], shape=(0, 3), dtype=object)